In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '2_train_clean2.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [5]:
# ============================================================
# 3. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_fe = train[num_cols_fe]
X_val_fe = val[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])
# 여기까지는 일반적은 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_fe = hstack([
    X_train_fe,
    train_st_id_ohe
]).tocsr()  # df + 희소행력(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_with_station_fe = hstack([
    X_val_fe,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_fe.shape, X_train_with_station_fe.shape)
print(X_val_fe.shape, X_val_with_station_fe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(616299, 31) (616299, 496)
(133325, 31) (133325, 496)


In [6]:
# ============================================================
# 4. 모델비교
# ============================================================

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

# 모델 구동시간 측정용
import time

# 모델 정의
models = {
    'LightGBM': LGBMRegressor(
        random_state = 42
    ),
    'XGBoost' : XGBRegressor(
    random_state = 42
    ),
    'CatBoost' : CatBoostRegressor(
    random_state = 42
    ),
}

# 학습, 예측, 평가
results = {}

for name, model in models.items():

    # 학습
    model.fit(
        X_train_with_station_fe,
        y_train
    )

    # 예측
    pred = model.predict(X_val_with_station_fe)

    # RMSE 평가
    rmse = root_mean_squared_error(
        y_val,
        pred
    )

    results[name] = round(rmse, 4)
    print()


result_df = pd.DataFrame(
    results.items(),
    columns = ['model', 'RMSE']
)

print(result_df)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015248 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8757
[LightGBM] [Info] Number of data points in the train set: 616299, number of used features: 496
[LightGBM] [Info] Start training from score 0.111706


Learning rate set to 0.112971
0:	learn: 0.3821094	total: 242ms	remaining: 4m 1s
1:	learn: 0.3645451	total: 294ms	remaining: 2m 26s
2:	learn: 0.3497160	total: 339ms	remaining: 1m 52s
3:	learn: 0.3371944	total: 383ms	remaining: 1m 35s
4:	learn: 0.3266016	total: 432ms	remaining: 1m 25s
5:	learn: 0.3176704	total: 474ms	remaining: 1m 18s
6:	learn: 0.3096941	total: 531ms	remaining: 1m 15s
7:	learn: 0.3028641	total: 593ms	remaining: 1m 13s
8:	learn: 0.2972424	total: 641ms	remaining: 1m 10s
9:	learn: 0.2923176	total: 687ms	remaining: 1m 8s
10:	learn: 0.2881155	total: 737ms	remaining: 1m 6s
1